# This notebook hosts the training loops for part 1 and part 2

# Part 1

In [ ]:

# Check GPU is enabled
import torch
import numpy as np
import matplotlib.pyplot as plt

print("="*60)
print("SYSTEM CHECK")
print("="*60)
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = 'cuda'
else:
    print("⚠️ WARNING: No GPU detected!")
    print("Go to: Runtime → Change runtime type → Hardware accelerator → GPU → Save")
    print("Then: Runtime → Restart runtime")
    device = 'cpu'
print("="*60 + "\n")

# import modules 
try:
    from dataloader import create_meshgrid, normalize_color, read_img
    from positional_encoding import positional_encoding
    from network import NeuralField2D
    import importlib
    import train

    # Reload the module
    importlib.reload(train)

    # Now use the updated functions
    from train import train_2D_neural_field, render_img, visualize_training_run
    print(" All modules imported successfully!\n")
except ImportError as e:
    print(f"ERROR: Could not import modules!")
    print(f"Error message: {e}")
    raise

# Set hyperparams
print("="*60)
print("HYPERPARAMETERS")
print("="*60)
L = 1
num_iterations = 5000
batch_size = 10000
learning_rate = 1e-2
log_every = 100
visualize_every = 500  # Set to None to disable visualization

print(f"Device: {device}")
print(f"Positional encoding L: {L}")
print(f"Iterations: {num_iterations:,}")
print(f"Batch size: {batch_size:,}")
print(f"Learning rate: {learning_rate}")
print(f"Visualize every: {visualize_every} iterations")
print("="*60 + "\n")

# Load and preprocess image
print("Loading image...")
img_path = "adrian.jpg"  

try:
    image = read_img(img_path)
    h, w = image.shape[:2]
    print(f"Image loaded: {w}x{h} pixels ({h*w:,} total pixels)\n")
except FileNotFoundError:
    print(f" ERROR: Could not find '{img_path}'")
    raise

# Create coordinate grid and normalize colors
coords = create_meshgrid(h, w)
colors = normalize_color(image)

# apply positional encoding
print(f"Applying positional encoding with L={L}...")
encoded_coords = positional_encoding(coords, L)


# create model
input_dim = encoded_coords.shape[1]
print(f"Creating neural network...")
model = NeuralField2D(input_dim=input_dim, hidden_dim=256, num_layers=4)
print()

# TRAIN THE MODEL!
print("="*60)
print("STARTING TRAINING")
print("="*60 + "\n")

loss_log, psnr_log = train_2D_neural_field(
    model=model,
    coords=encoded_coords,
    colors=colors,
    num_iterations=num_iterations,
    batch_size=batch_size,
    learning_rate=learning_rate,
    device=device,
    log_every=log_every,
    visualize_every=visualize_every,
    height=h,
    width=w,
    original_img=image
)

# Render final high-quality reconstruction
print("\n" + "="*60)
print("GENERATING FINAL RECONSTRUCTION")
print("="*60)
encoded_coords = torch.tensor(encoded_coords, dtype=torch.float32, device=device)
reconstructed_img = render_img(model, h, w, encoded_coords, device)
print("Reconstruction complete!\n")

# Visualize results
print("Generating visualization plots...")
visualize_training_run(loss_log, psnr_log, image, reconstructed_img)

# Save outputs
print("\n" + "="*60)
print("SAVING OUTPUTS")
print("="*60)

# Save model weights
model_path = 'neural_field_2d.pth'
torch.save(model.state_dict(), model_path)
print(f"✓ Model saved to '{model_path}'")

# Save reconstructed image
from PIL import Image
reconstructed_uint8 = (np.clip(reconstructed_img, 0, 1) * 255).astype(np.uint8)
output_path = 'reconstructed_fox.png'
Image.fromarray(reconstructed_uint8).save(output_path)
print(f"✓ Reconstructed image saved to '{output_path}'")

# Save loss/PSNR curves
np.savez('training_metrics.npz', loss=loss_log, psnr=psnr_log)
print(f"✓ Training metrics saved to 'training_metrics.npz'")

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"Final PSNR: {psnr_log[-1]:.2f} dB")
print("="*60)

# Part 2

In [ ]:
#3D NeRF Training Script for Lego Dataset

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
from PIL import Image
import imageio

# system check
print("="*60)
print("3D NeRF TRAINING - LEGO DATASET")
print("="*60)
print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    device = 'cuda'
else:
    print("Running on CPU (will be slow!)")
    device = 'cpu'
print("="*60 + "\n")


# import modules
from dataloader import load_lego_data
from utils import create_rays_dataset, sample_rays, sample_points_along_rays
from network import NeRF, prepare_nerf_inputs
from volume_rendering import volume_rendering

#load lego data
print("Loading Lego dataset...")
images_train, images_val, c2ws_train, c2ws_val, c2ws_test, focal = load_lego_data()

H, W = images_train.shape[1:3]
focal = float(focal)

print(f"  Training images: {len(images_train)} ({H}x{W})")
print(f"  Validation images: {len(images_val)}")
print(f"  Test cameras: {len(c2ws_test)}")
print(f"  Focal length: {focal:.2f}")

# Create intrinsic matrix
K = torch.tensor([
    [focal, 0.0, W/2],
    [0.0, focal, H/2],
    [0.0, 0.0, 1.0]
])

# precompute all rays
print("\nPrecomputing rays from training images...")
rays_o_train, rays_d_train, colors_train, uvs_train = create_rays_dataset(
    images_train, K, c2ws_train
)

print("\nPrecomputing rays from validation images...")
rays_o_val, rays_d_val, colors_val, uvs_val = create_rays_dataset(
    images_val, K, c2ws_val
)

# visualize cameras and rays (deliverable)
def show_cameras_and_rays(images, c2ws, K, rays_o_all, rays_d_all, num_rays=100):
    """
    Visualize cameras and sampled rays using viser.
    DELIVERABLE: Visualization of rays and cameras.
    """
    import viser
    
    print("\n" + "="*60)
    print("DELIVERABLE 1: Visualizing Cameras and Rays")
    print("="*60)
    
    # Sample rays
    indices = torch.randint(0, rays_o_all.shape[0], (num_rays,))
    rays_o = rays_o_all[indices]
    rays_d = rays_d_all[indices]
    
    # Sample points along rays
    points, _ = sample_points_along_rays(
        rays_o, rays_d, near=2.0, far=6.0, n_samples=64, perturb=False
    )
    
    # Convert to numpy
    if isinstance(rays_o, torch.Tensor):
        rays_o = rays_o.numpy()
        rays_d = rays_d.numpy()
        points = points.numpy()
    
    # Create viser server
    server = viser.ViserServer(share=True)
    
    # Add cameras
    for i, (image, c2w) in enumerate(zip(images, c2ws)):
        server.scene.add_camera_frustum(
            f"/cameras/{i}",
            fov=2 * np.arctan2(H / 2, K[0, 0]),
            aspect=W / H,
            scale=0.15,
            wxyz=viser.transforms.SO3.from_matrix(c2w[:3, :3]).wxyz,
            position=c2w[:3, 3],
            image=image
        )
    
    # Add rays
    for i, (o, d) in enumerate(zip(rays_o, rays_d)):
        server.scene.add_spline_catmull_rom(
            f"/rays/{i}",
            positions=np.stack((o, o + d * 6.0)),
        )
    
    # Add sample points
    server.scene.add_point_cloud(
        "/samples",
        colors=np.zeros_like(points).reshape(-1, 3),
        points=points.reshape(-1, 3),
        point_size=0.02,
    )
    
    print("\nVisualization ready!")
    print("Open the URL above in your browser and take screenshot hten ctrlc")

    
    try:
        while True:
            time.sleep(0.1)
    except KeyboardInterrupt:
        print("\nContinuing to training...\n")


# Uncomment to visualize (will pause execution)
# show_cameras_and_rays(images_train, c2ws_train, K, rays_o_train, rays_d_train, num_rays=100)


# training fn
def train_nerf(model, rays_o_train, rays_d_train, colors_train,
               rays_o_val, rays_d_val, colors_val, images_val,
               num_iterations=1000, batch_size=10000, learning_rate=5e-4,
               device='cuda', log_every=100, render_every=200,
               near=2.0, far=6.0, n_samples=64):
    """
    Train NeRF on Lego dataset.
    
    DELIVERABLE: Training visualization and PSNR curves.
    """
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    
    # Move data to device
    rays_o_train = rays_o_train.to(device)
    rays_d_train = rays_d_train.to(device)
    colors_train = colors_train.to(device)
    
    # Tracking
    train_psnrs = []
    val_psnrs = []
    iterations_logged = []
    rendered_images = []  # Store rendered validation images
    
    print("="*60)
    print("STARTING TRAINING")
    print("="*60)
    print(f"Device: {device}")
    print(f"Iterations: {num_iterations}")
    print(f"Batch size: {batch_size} rays")
    print(f"Learning rate: {learning_rate}")
    print(f"Samples per ray: {n_samples}")
    print("="*60 + "\n")
    
    start_time = time.time()
    
    for iteration in tqdm(range(num_iterations), desc="Training"):
        model.train()
        
        # Sample random rays
        indices = torch.randint(0, rays_o_train.shape[0], (batch_size,), device=device)
        batch_rays_o = rays_o_train[indices]
        batch_rays_d = rays_d_train[indices]
        batch_colors_gt = colors_train[indices]
        
        # Sample points along rays
        points, t_vals = sample_points_along_rays(
            batch_rays_o, batch_rays_d, near=near, far=far, 
            n_samples=n_samples, perturb=True
        )
        
        # Prepare NeRF inputs
        pos_enc, dir_enc = prepare_nerf_inputs(points, batch_rays_d, L_pos=10, L_dir=4)
        
        # Forward pass through NeRF
        rgb_pred, sigma_pred = model(pos_enc, dir_enc)
        
        # Volume rendering
        rendered_colors = volume_rendering(rgb_pred, sigma_pred, t_vals)
        
        # Compute loss
        loss = torch.mean((rendered_colors - batch_colors_gt) ** 2)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Compute PSNR
        with torch.no_grad():
            psnr = -10.0 * torch.log10(loss)
            train_psnrs.append(psnr.item())
        
        # Logging
        if (iteration + 1) % log_every == 0 or iteration == 0:
            elapsed = time.time() - start_time
            print(f"Iter {iteration+1}/{num_iterations} | "
                  f"Loss: {loss.item():.6f} | "
                  f"PSNR: {psnr.item():.2f} dB | "
                  f"Time: {elapsed:.1f}s")
        
        # Validation and rendering (LIVE VIEW)
        if (iteration + 1) % render_every == 0 or iteration == 0:
            model.eval()
            with torch.no_grad():
                val_psnr = evaluate_on_validation(
                    model, rays_o_val, rays_d_val, colors_val,
                    near, far, n_samples, device
                )
            val_psnrs.append(val_psnr)
            iterations_logged.append(iteration + 1)

            # Render one validation image
            rendered_img = render_full_image(
                model, images_val[0], c2ws_val[0], K, H, W,
                near, far, n_samples, device
            )
            rendered_images.append((iteration + 1, rendered_img))

            # live display
            clear_output(wait=True)

            fig = plt.figure(figsize=(12,5))

            # Left: training PSNR (so far)
            ax1 = fig.add_subplot(1,2,1)
            ax1.plot(train_psnrs, linewidth=0.8)
            if len(train_psnrs) > 50:
                smoothed = np.convolve(train_psnrs, np.ones(50)/50, mode='valid')
                ax1.plot(range(49, len(train_psnrs)), smoothed, linewidth=2)
            ax1.set_title("Training PSNR")
            ax1.set_xlabel("Iteration")
            ax1.set_ylabel("PSNR (dB)")

            # right: rendered validation image
            ax2 = fig.add_subplot(1,2,2)
            ax2.imshow(np.clip(rendered_img, 0, 1))
            ax2.set_title(f"Rendered @ Iter {iteration+1}")
            ax2.axis("off")

            display(fig)
            plt.close(fig)

            print(f"[Iter {iteration+1}] Train PSNR: {train_psnrs[-1]:.2f}, Val PSNR: {val_psnr:.2f}")
    
    total_time = time.time() - start_time
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print(f"Total time: {total_time/60:.1f} minutes")
    print(f"Final training PSNR: {train_psnrs[-1]:.2f} dB")
    print(f"Final validation PSNR: {val_psnrs[-1]:.2f} dB")
    print("="*60)
    
    return train_psnrs, val_psnrs, iterations_logged, rendered_images


def evaluate_on_validation(model, rays_o_val, rays_d_val, colors_val,
                           near, far, n_samples, device):
    """Evaluate on validation set."""
    model.eval()
    
    # Use subset for faster evaluation
    num_eval_rays = min(10000, rays_o_val.shape[0])
    indices = torch.randint(0, rays_o_val.shape[0], (num_eval_rays,))
    
    rays_o = rays_o_val[indices].to(device)
    rays_d = rays_d_val[indices].to(device)
    colors_gt = colors_val[indices].to(device)
    
    with torch.no_grad():
        # Sample points
        points, t_vals = sample_points_along_rays(
            rays_o, rays_d, near=near, far=far, 
            n_samples=n_samples, perturb=False
        )
        
        # Prepare inputs
        pos_enc, dir_enc = prepare_nerf_inputs(points, rays_d, L_pos=10, L_dir=4)
        
        # Forward pass
        rgb_pred, sigma_pred = model(pos_enc, dir_enc)
        
        # Volume rendering
        rendered_colors = volume_rendering(rgb_pred, sigma_pred, t_vals)
        
        # Compute PSNR
        mse = torch.mean((rendered_colors - colors_gt) ** 2)
        psnr = -10.0 * torch.log10(mse)
    
    return psnr.item()


def render_full_image(model, image, c2w, K, H, W, near, far, n_samples, device):
    """Render a complete image from a camera pose."""
    model.eval()
    
    # Create all rays for this camera
    from utils import pixel_to_ray
    
    u_coords = torch.arange(W, dtype=torch.float32) + 0.5
    v_coords = torch.arange(H, dtype=torch.float32) + 0.5
    u_grid, v_grid = torch.meshgrid(u_coords, v_coords, indexing='xy')
    uvs = torch.stack([u_grid.flatten(), v_grid.flatten()], dim=-1)
    
    c2w_tensor = torch.from_numpy(c2w).float()
    rays_o, rays_d = pixel_to_ray(K, c2w_tensor, uvs)
    
    # Render in batches
    batch_size = 1000
    rendered_pixels = []
    
    with torch.no_grad():
        for i in range(0, rays_o.shape[0], batch_size):
            batch_rays_o = rays_o[i:i+batch_size].to(device)
            batch_rays_d = rays_d[i:i+batch_size].to(device)
            
            # Sample points
            points, t_vals = sample_points_along_rays(
                batch_rays_o, batch_rays_d, near=near, far=far,
                n_samples=n_samples, perturb=False
            )
            
            # Prepare inputs
            pos_enc, dir_enc = prepare_nerf_inputs(points, batch_rays_d, L_pos=10, L_dir=4)
            
            # Forward pass
            rgb_pred, sigma_pred = model(pos_enc, dir_enc)
            
            # Volume rendering
            rendered = volume_rendering(rgb_pred, sigma_pred, t_vals)
            rendered_pixels.append(rendered.cpu())
    
    # Reshape to image
    rendered_pixels = torch.cat(rendered_pixels, dim=0)
    rendered_image = rendered_pixels.reshape(H, W, 3).numpy()
    
    return rendered_image


# train the model
print("\ncreating NeRF model...")
model = NeRF(pos_enc_dim=63, dir_enc_dim=27)

print("\n starting training...")
train_psnrs, val_psnrs, iterations_logged, rendered_images = train_nerf(
    model,
    rays_o_train, rays_d_train, colors_train,
    rays_o_val, rays_d_val, colors_val, images_val,
    num_iterations=2000,
    batch_size=10000,
    learning_rate=5e-4,
    device=device,
    log_every=50,
    render_every=200,
    near=2.0,
    far=6.0,
    n_samples=64
)


# visualize training progress deliverable
def visualize_training_progress(train_psnrs, val_psnrs, iterations_logged, 
                                rendered_images, images_val):
    """
    DELIVERABLE: visualize training curves and rendered images.
    """
    print("\n" + "="*60)
    print("DELIVERABLE: Training Visualization")
    print("="*60)
    
    fig = plt.figure(figsize=(18, 12))
    
    # Plot 1: Training PSNR curve
    ax1 = plt.subplot(3, 3, 1)
    ax1.plot(train_psnrs, alpha=0.7, linewidth=0.5)
    # Moving average
    window = 50
    if len(train_psnrs) > window:
        train_psnrs_smooth = np.convolve(train_psnrs, np.ones(window)/window, mode='valid')
        ax1.plot(range(window-1, len(train_psnrs)), train_psnrs_smooth, 
                linewidth=2, label='Moving avg')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('PSNR (dB)')
    ax1.set_title('Training PSNR')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Plot 2: Validation PSNR curve
    ax2 = plt.subplot(3, 3, 2)
    ax2.plot(iterations_logged, val_psnrs, 'o-', linewidth=2, markersize=6)
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('PSNR (dB)')
    ax2.set_title('Validation PSNR')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=23, color='r', linestyle='--', label='Target: 23 dB')
    ax2.legend()
    
    # Plot 3-8: Rendered images at different iterations
    for idx, (iteration, rendered_img) in enumerate(rendered_images):
        ax = plt.subplot(3, 3, idx + 3)
        ax.imshow(np.clip(rendered_img, 0, 1))
        ax.set_title(f'Iter {iteration}')
        ax.axis('off')
    
    # Ground truth
    if idx + 4 < 9:
        ax = plt.subplot(3, 3, idx + 4)
        ax.imshow(images_val[0])
        ax.set_title('Ground Truth')
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('training_progress.png', dpi=150, bbox_inches='tight')
    print("✓ Saved training progress to 'training_progress.png'")
    plt.show()


visualize_training_progress(train_psnrs, val_psnrs, iterations_logged, 
                           rendered_images, images_val)


#render novel view video (deliverable)
def render_novel_view_video(model, c2ws_test, K, H, W, near, far, n_samples, device):
    """
    DELIVERABLE 3: Render novel views from test cameras.
    """
    print("\n" + "="*60)
    print("DELIVERABLE: Rendering Novel View Video")
    print("="*60)
    
    frames = []
    
    for i, c2w in enumerate(tqdm(c2ws_test, desc="Rendering frames")):
        frame = render_full_image(
            model, None, c2w, K, H, W, near, far, n_samples, device
        )
        # Convert to uint8
        frame_uint8 = (np.clip(frame, 0, 1) * 255).astype(np.uint8)
        frames.append(frame_uint8)
    
    # Save as GIF
    imageio.mimsave('novel_view_video.gif', frames, fps=10, loop=0)
    print("Saved novel view video to 'novel_view_video.gif'")
    
    # Save as MP4 (if ffmpeg available)
    try:
        imageio.mimsave('novel_view_video.mp4', frames, fps=10)
        print("Saved novel view video to 'novel_view_video.mp4'")
    except:
        print("(MP4 save failed - ffmpeg not available)")
    
    return frames


frames = render_novel_view_video(model, c2ws_test, K, H, W, 2.0, 6.0, 64, device)


# save model and outputs
print("\n" + "="*60)
print("SAVING OUTPUTS")
print("="*60)

torch.save(model.state_dict(), 'nerf_lego.pth')
print("✓ Model saved to 'nerf_lego.pth'")

np.savez('training_metrics.npz', 
         train_psnr=train_psnrs, 
         val_psnr=val_psnrs,
         iterations=iterations_logged)
print("✓ Training metrics saved")

print("\n" + "="*60)
print("ALL DELIVERABLES COMPLETE! YIPEEEEEE thank goodness")
print("="*60)
print("\nDeliverables:")
print("1. Camera and ray visualization (take screenshot from viser)")
print("2. Training progress visualization (training_progress.png)")
print("3. Novel view video (novel_view_video.gif)")
print(f"\nFinal validation PSNR: {val_psnrs[-1]:.2f} dB")
print("="*60)